# Wave Field LLM — Long Context Scaling Test (Colab T4 16GB)

**Goal:** Prove O(n log n) advantage at n=4096 and n=8192 where standard attention's O(n²) actually hurts.

At n=512: wave is 2x slower (FFT overhead > attention savings). We need n≥4096.

| Seq Len | Theoretical Speedup | What we need to prove |
|---------|--------------------|-----------------------|
| 512 | 1.2x | ❌ Useless |
| 2048 | 1.8x | ❌ Still slower (tested) |
| 4096 | 2.7x | ⬅️ Crossover point? |
| 8192 | 4.4x | ⬅️ Wave should win here |

In [ ]:
# Cell 1: Setup — clone repo, install deps
!git clone https://github.com/Pankh-AI/wave-field-llm.git
%cd wave-field-llm
!pip install -q torch datasets tokenizers matplotlib wandb

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Cell 2: VRAM calculator — find max batch size per seq length
import torch
import os, gc
os.environ['SPECTRAL_GATE'] = '1'

from src.wave_field_transformer import WaveFieldTransformer

torch.set_float32_matmul_precision('high')
device = 'cuda'

def estimate_max_batch(seq_len, field_size, model_type='wave', target_vram_gb=14.5):
    """Binary search for max batch size that fits in VRAM."""
    torch.cuda.empty_cache()
    gc.collect()
    
    for batch_size in [16, 12, 8, 6, 4, 3, 2, 1]:
        try:
            torch.cuda.empty_cache()
            if model_type == 'wave':
                os.environ['HYBRID_LAYERS'] = '3'
                model = WaveFieldTransformer(
                    vocab_size=8000, embedding_dim=384, num_layers=8, num_heads=8,
                    ffn_dim=1536, field_size=field_size, max_seq_len=seq_len+2,
                    dropout=0.1, hybrid_attention_layers=[3], device=device
                ).to(device)
            else:
                model = torch.nn.TransformerEncoder(
                    torch.nn.TransformerEncoderLayer(
                        d_model=384, nhead=8, dim_feedforward=1536,
                        dropout=0.1, activation='gelu', batch_first=True, norm_first=True
                    ), num_layers=8
                ).to(device)
            
            ids = torch.randint(0, 8000, (batch_size, seq_len), device=device)
            with torch.amp.autocast(device, dtype=torch.bfloat16):
                if model_type == 'wave':
                    logits, loss = model(ids, labels=ids)
                else:
                    x = torch.randn(batch_size, seq_len, 384, device=device)
                    mask = torch.nn.Transformer.generate_square_subsequent_mask(seq_len, device=device)
                    out = model(x, mask=mask, is_causal=True)
                    loss = out.sum()
                loss.backward()
            
            vram_used = torch.cuda.max_memory_allocated() / 1e9
            del model, ids, loss
            if 'logits' in dir(): del logits
            if 'out' in dir(): del out
            torch.cuda.empty_cache()
            gc.collect()
            
            if vram_used < target_vram_gb:
                print(f'  {model_type:>8s} seq={seq_len:>5d} batch={batch_size:>2d} → {vram_used:.1f}GB ✓')
                return batch_size
        except (torch.cuda.OutOfMemoryError, RuntimeError):
            del model
            torch.cuda.empty_cache()
            gc.collect()
            continue
    return 0

print('Finding max batch sizes (target <14.5GB):\n')
configs = {}
for seq_len in [2048, 4096, 8192]:
    print(f'--- seq_len={seq_len} ---')
    wb = estimate_max_batch(seq_len, seq_len, 'wave')
    sb = estimate_max_batch(seq_len, seq_len, 'standard')
    configs[seq_len] = {'wave_batch': wb, 'std_batch': sb}
    print()

print('\nConfigs found:')
for sl, cfg in configs.items():
    print(f'  n={sl}: wave batch={cfg["wave_batch"]}, std batch={cfg["std_batch"]}')

In [ ]:
# Cell 3: Speed benchmark — Wave vs Standard at each seq length
import time
import torch
import os, gc
os.environ['SPECTRAL_GATE'] = '1'

from src.wave_field_transformer import WaveFieldTransformer

torch.set_float32_matmul_precision('high')
device = 'cuda'

def speed_bench(model_type, seq_len, field_size, batch_size, n_steps=20, warmup=5):
    torch.cuda.empty_cache()
    gc.collect()
    torch.manual_seed(42)
    
    if model_type == 'wave':
        model = WaveFieldTransformer(
            vocab_size=8000, embedding_dim=384, num_layers=8, num_heads=8,
            ffn_dim=1536, field_size=field_size, max_seq_len=seq_len+2,
            dropout=0.1, hybrid_attention_layers=[3], use_checkpoint=True,
            device=device
        ).to(device)
        optimizer = model.configure_optimizer(base_lr=3e-4, kernel_lr_mult=15.0)
    else:
        from src.wave_field_transformer import WaveFieldTransformer as WFT
        model = WFT(
            vocab_size=8000, embedding_dim=384, num_layers=8, num_heads=8,
            ffn_dim=1536, field_size=field_size, max_seq_len=seq_len+2,
            dropout=0.1, use_checkpoint=True, device=device
        ).to(device)
        optimizer = model.configure_optimizer(base_lr=3e-4)
    
    scaler = torch.amp.GradScaler(device)
    ids = torch.randint(0, 8000, (batch_size, seq_len), device=device)
    
    # Warmup
    for _ in range(warmup):
        optimizer.zero_grad()
        with torch.amp.autocast(device, dtype=torch.bfloat16):
            _, loss = model(ids, labels=ids)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        optimizer.zero_grad()
        with torch.amp.autocast(device, dtype=torch.bfloat16):
            _, loss = model(ids, labels=ids)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    
    tps = n_steps * batch_size * seq_len / elapsed
    vram = torch.cuda.max_memory_allocated() / 1e9
    
    del model, optimizer, scaler, ids
    torch.cuda.empty_cache()
    gc.collect()
    
    return tps, vram, elapsed

# Run benchmarks
# Adjust batch sizes based on Cell 2 results, or use conservative defaults
test_configs = [
    # (seq_len, wave_batch, std_batch)
    (512,  16, 16),
    (2048,  4,  4),
    (4096,  2,  2),
    (8192,  1,  1),
]

print(f'{"Seq":>6s} {"Wave tok/s":>12s} {"Std tok/s":>12s} {"Speedup":>8s} {"Wave VRAM":>10s} {"Std VRAM":>10s}')
print('-' * 65)

results = []
for seq_len, wb, sb in test_configs:
    print(f'\nTesting n={seq_len}...')
    try:
        w_tps, w_vram, _ = speed_bench('wave', seq_len, seq_len, wb)
    except Exception as e:
        print(f'  Wave failed: {e}')
        w_tps, w_vram = 0, 0
    try:
        s_tps, s_vram, _ = speed_bench('standard', seq_len, seq_len, sb)
    except Exception as e:
        print(f'  Standard failed: {e}')
        s_tps, s_vram = 0, 0
    
    speedup = w_tps / s_tps if s_tps > 0 else 0
    print(f'{seq_len:>6d} {w_tps:>12,.0f} {s_tps:>12,.0f} {speedup:>7.2f}x {w_vram:>9.1f}G {s_vram:>9.1f}G')
    results.append({'seq_len': seq_len, 'wave_tps': w_tps, 'std_tps': s_tps, 'speedup': speedup})

print('\n' + '=' * 65)
print('SUMMARY')
print('=' * 65)
for r in results:
    status = '✅ WAVE WINS' if r['speedup'] > 1.0 else '❌ std faster'
    print(f'  n={r["seq_len"]:>5d}: wave {r["wave_tps"]:>8,.0f} vs std {r["std_tps"]:>8,.0f} = {r["speedup"]:.2f}x  {status}')

In [ ]:
# Cell 4: Full training run at the crossover point
# Run this AFTER Cell 3 identifies where wave beats standard on speed
import os, subprocess, sys

# Set the seq length where wave wins (update based on Cell 3)
SEQ_LEN = 4096  # <-- update this based on Cell 3 results
BATCH = 2       # <-- update based on Cell 2

os.environ.update({
    'SPECTRAL_GATE': '1',
    'HYBRID_LAYERS': '3',
    'KERNEL_LR_MULT': '15.0',
    'MONITOR_SNAPSHOTS': '3',
    'USE_CHECKPOINT': '1',
})

# Add S1-4K config dynamically
sys.path.insert(0, '.')
from benchmarks.benchmark_scaling import SCALE_CONFIGS

SCALE_CONFIGS[f'S1-{SEQ_LEN//1024}K'] = {
    'name': f'S1-{SEQ_LEN//1024}K (22M / 20M tok / seq={SEQ_LEN})',
    'embedding_dim': 384,
    'num_layers': 8,
    'num_heads': 8,
    'ffn_dim': 1536,
    'field_size': SEQ_LEN,
    'seq_len': SEQ_LEN,
    'batch_size': BATCH,
    'token_budget': 20_000_000,
    'peak_lr': 3e-4,
    'use_checkpoint': True,
}

os.environ['SCALE'] = f'S1-{SEQ_LEN//1024}K'
os.environ['DATASET'] = '2'

print(f'Running both models at n={SEQ_LEN}, batch={BATCH}')
print(f'This will take ~30-60 min depending on seq length')
print()

!python -u benchmarks/benchmark_scaling.py

In [ ]:
# Cell 5: Plot results
import matplotlib.pyplot as plt
import json, os

# Load results if available
results_dir = 'results/data'
if os.path.exists(results_dir):
    for f in sorted(os.listdir(results_dir)):
        if f.endswith('.json'):
            print(f'Found: {f}')

# Plot speed comparison from Cell 3
if 'results' in dir() and results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    seq_lens = [r['seq_len'] for r in results if r['wave_tps'] > 0]
    wave_tps = [r['wave_tps'] for r in results if r['wave_tps'] > 0]
    std_tps = [r['std_tps'] for r in results if r['wave_tps'] > 0]
    speedups = [r['speedup'] for r in results if r['wave_tps'] > 0]
    
    ax1.plot(seq_lens, wave_tps, 'o-', label='Wave Field', color='#14a3a8', linewidth=2)
    ax1.plot(seq_lens, std_tps, 's-', label='Standard', color='#e74c3c', linewidth=2)
    ax1.set_xlabel('Sequence Length')
    ax1.set_ylabel('Tokens/sec')
    ax1.set_title('Speed: Wave vs Standard')
    ax1.legend()
    ax1.set_xscale('log', base=2)
    ax1.grid(True, alpha=0.3)
    
    ax2.bar(range(len(seq_lens)), speedups, color=['#e74c3c' if s < 1 else '#2ecc71' for s in speedups])
    ax2.axhline(y=1.0, color='black', linestyle='--', label='Breakeven')
    ax2.set_xticks(range(len(seq_lens)))
    ax2.set_xticklabels([str(s) for s in seq_lens])
    ax2.set_xlabel('Sequence Length')
    ax2.set_ylabel('Speedup (Wave / Standard)')
    ax2.set_title('Speed Crossover')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/speed_crossover.png', dpi=150)
    plt.show()
    print('Saved: results/speed_crossover.png')